In [279]:
# ============================================================
# benchmark_fetal_plane.py
# Paper-style benchmark:
# USFM, MOFO, UltraSAM, FetalCLIP, ResNet50, EfficientNetV2, DINOv3
# Linear Probing + Full Fine-Tuning
# ============================================================

import os
import re
import copy
import numpy as np
from omegaconf import OmegaConf
# import sys

# sys.path.append(
#     r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\USFM"
# )
# from usdsgen.modules.backbone.vision_transformer import build_vit
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score



In [280]:
# ============================================================
# 1. CONFIGURATION
# ============================================================


IMAGE_ROOT =r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\Image_F"


USF_MAE_CKPT = r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\USF-MAE\mae_US_saved_models\USF-MAE_fold1_lr0.0003_wd0.01_epochs100.pt"
ULTRASAM_ROOT = r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\UltraSam"
ULTRASAM_CKPT = r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\UltraSam.pth"

MOFO_ROOT = r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\MOFO"
MOFO_CKPT = r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\mofo_weight_sel.pth"

NUM_CLASSES = 6
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_FOLDS = 5
NUM_WORKERS = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [281]:
def extract_patient_id(image_path):
    name = os.path.basename(image_path)

    match = re.search(r"(Patient\d+)", name, re.IGNORECASE)
    if match:
        return match.group(1)

    return os.path.splitext(name)[0]


class FolderFetalPlaneDataset(Dataset):
    def __init__(self, image_root, transform=None):
        self.image_root = image_root
        self.transform = transform

        CLASS_MAP = {
            "Fetal abdomen": 0,
            "Fetal brain": 1,
            "Fetal femur": 2,
            "Fetal thorax": 3,
            "Maternal cervix": 4,
            "Other": 5
        }

        self.class_names = list(CLASS_MAP.keys())
        self.class_to_idx = CLASS_MAP

        self.samples = []
        valid_exts = (".png", ".jpg", ".jpeg", ".bmp")

        for class_name in self.class_names:
            class_folder = os.path.join(image_root, class_name)
            if not os.path.isdir(class_folder):
                print(f"Warning: class folder not found: {class_folder}")
                continue

            for root_dir, _, files in os.walk(class_folder):
                for file_name in files:
                    if file_name.lower().endswith(valid_exts):
                        image_path = os.path.join(root_dir, file_name)
                        label = self.class_to_idx[class_name]
                        self.samples.append((image_path, label, class_name))

        if len(self.samples) == 0:
            print(f"Warning: no images found in {image_root}. Check nested folders or file extensions.")

        print("Classes:", self.class_to_idx)
        print("Total images:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label, class_name = self.samples[idx]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


In [282]:
# ============================================================
# 3. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])


In [283]:
# ============================================================
# 4. CLASSIFICATION WRAPPER
# ============================================================

class EncoderClassifier(nn.Module):
    def __init__(self, encoder, feature_dim, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        features = self.encoder(x)

        if isinstance(features, dict):
            if "pooler_output" in features:
                features = features["pooler_output"]
            elif "last_hidden_state" in features:
                features = features["last_hidden_state"][:, 0, :]
            elif "features" in features:
                features = features["features"]

        if isinstance(features, (tuple, list)):
            features = features[0]

        if features.ndim == 4:
            features = torch.mean(features, dim=[2, 3])

        if features.ndim == 3:
            features = features[:, 0, :]

        return self.classifier(features)


In [284]:
# ============================================================
# BASELINE ENCODERS
# ============================================================

class ResNet50Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.feature_dim = model.fc.in_features
        model.fc = nn.Identity()
        self.encoder = model

    def forward(self, x):
        return self.encoder(x)


class EfficientNetV2Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = models.efficientnet_v2_s(
            weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        )
        self.feature_dim = model.classifier[1].in_features
        model.classifier = nn.Identity()
        self.encoder = model

    def forward(self, x):
        return self.encoder(x)


class DINOEncoder(nn.Module):
    def __init__(self, model_name="facebook/dinov2-base"):
        super().__init__()
        from transformers import AutoModel
        self.encoder = AutoModel.from_pretrained(model_name)
        self.feature_dim = self.encoder.config.hidden_size

    def forward(self, x):
        out = self.encoder(pixel_values=x)
        return out.last_hidden_state[:, 0, :]


In [285]:
# ============================================================
# 6. FOUNDATION MODEL ADAPTERS
# ============================================================
## USFM model 

class USFMAEEncoder(nn.Module):
    def __init__(self, checkpoint_path):
        super().__init__()
        import timm

        self.encoder = timm.create_model(
            "vit_base_patch16_224",
            pretrained=False,
            num_classes=0
        )

        ckpt = torch.load(checkpoint_path, map_location="cpu")

        if isinstance(ckpt, dict):
            if "model" in ckpt:
                ckpt = ckpt["model"]
            elif "state_dict" in ckpt:
                ckpt = ckpt["state_dict"]

        clean_ckpt = {}

        for k, v in ckpt.items():
            k = k.replace("module.", "")
            k = k.replace("encoder.", "")
            k = k.replace("mae.", "")

            if k.startswith("decoder"):
                continue
            if k.startswith("mask_token"):
                continue

            clean_ckpt[k] = v

        missing, unexpected = self.encoder.load_state_dict(clean_ckpt, strict=False)

        print("USF-MAE missing:", len(missing))
        print("USF-MAE unexpected:", len(unexpected))

        self.feature_dim = 768

    def forward(self, x):
        features = self.encoder.forward_features(x)

        if features.ndim == 3:
            return features[:, 0, :]

        return features

In [286]:
 ## fetal clip encoder , fetal is not workign bcs checkpoints are not there . 

# class FetalCLIPEncoder(nn.Module):
#     def __init__(self, checkpoint_path=None):
#         super().__init__()
#         model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
#         self.feature_dim = model.heads.head.in_features
#         model.heads = nn.Identity()
#         self.encoder = model

#         if checkpoint_path is not None:
#             checkpoint = torch.load(checkpoint_path, map_location="cpu")
#             if isinstance(checkpoint, dict):
#                 if "model" in checkpoint:
#                     checkpoint = checkpoint["model"]
#                 elif "state_dict" in checkpoint:
#                     checkpoint = checkpoint["state_dict"]
#             self.encoder.load_state_dict(checkpoint, strict=False)

#     def forward(self, x):
#         return self.encoder(x)



In [287]:
#### UltraSamencoder 


class UltraSAMEncoder(nn.Module):
    def __init__(self, checkpoint_path):
        super().__init__()

        import sys
        import torch

        sys.path.append(
            r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\UltraSam"
        )

        from endosam.models.backbones.vit_sam import ViTSAMcls 

        self.encoder = ViTSAMcls(
            arch="base",
            img_size=224,
            patch_size=16,
            out_indices=[11],
            use_abs_pos=True,
            use_rel_pos=True,
            window_size=14,
            out_channels=0
        )

        ckpt = torch.load(checkpoint_path, map_location="cpu")

        if isinstance(ckpt, dict):
            if "state_dict" in ckpt:
                ckpt = ckpt["state_dict"]
            elif "model" in ckpt:
                ckpt = ckpt["model"]

        clean_ckpt = {}

        for k, v in ckpt.items():
            k = k.replace("module.", "")
            k = k.replace("backbone.", "")
            k = k.replace("image_encoder.", "")
            clean_ckpt[k] = v

        missing, unexpected = self.encoder.load_state_dict(
            clean_ckpt,
            strict=False
        )

        print("UltraSAM missing keys:", len(missing))
        print("UltraSAM unexpected keys:", len(unexpected))

        self.feature_dim = 768

    def forward(self, x):
        outs, feature_vector = self.encoder(x)

        return feature_vector

In [288]:
# import sys
# import torch
# import torch.nn as nn

# ### foundational model USF_MAE

# USF_MAE_ROOT = r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\USF-MAE"
# USF_MAE_CKPT = r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\USF-MAE\mae_US_saved_models\USF-MAE_fold1_lr0.0003_wd0.01_epochs100.pt"

# sys.path.append(USF_MAE_ROOT)

# from mae.models_mae import mae_vit_base_patch16_dec512d8b


# class USFMAEEncoder(nn.Module):
#     def __init__(self, checkpoint_path):
#         super().__init__()

#         self.mae = mae_vit_base_patch16_dec512d8b()

#         ckpt = torch.load(checkpoint_path, map_location="cpu")

#         if isinstance(ckpt, dict):
#             if "model" in ckpt:
#                 ckpt = ckpt["model"]
#             elif "state_dict" in ckpt:
#                 ckpt = ckpt["state_dict"]

#         missing, unexpected = self.mae.load_state_dict(ckpt, strict=False)

#         print("USF-MAE missing keys:", len(missing))
#         print("USF-MAE unexpected keys:", len(unexpected))

#         self.feature_dim = 768

#     def forward(self, x):
#         # Patchify + add position embeddings + pass through encoder blocks.
#         x = self.mae.patch_embed(x)

#         cls_token = self.mae.cls_token + self.mae.pos_embed[:, :1, :]
#         cls_tokens = cls_token.expand(x.shape[0], -1, -1)

#         x = x + self.mae.pos_embed[:, 1:, :]
#         x = torch.cat((cls_tokens, x), dim=1)

#         for blk in self.mae.blocks:
#             x = blk(x)

#         x = self.mae.norm(x)

#         cls_feature = x[:, 0, :]

#         return cls_feature

In [289]:
### MOFO foundational model 

class MOFOEncoder(nn.Module):
    def __init__(self, checkpoint_path):
        super().__init__()

        import sys
        import torch

        sys.path.append(
            r"C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\MOFO"
        )

        from model.MOFO import backbone_encoder

        self.encoder = backbone_encoder(
            patch_size=4,
            embed_dim=64,
            depth=[2, 4, 32, 2],
            split_size=[1, 2, 7, 7],
            num_heads=[2, 4, 8, 16],
            mlp_ratio=4.0
        )

        ckpt = torch.load(checkpoint_path, map_location="cpu")

        if isinstance(ckpt, dict):
            if "state_dict" in ckpt:
                ckpt = ckpt["state_dict"]
            elif "model" in ckpt:
                ckpt = ckpt["model"]

        clean_ckpt = {}

        for k, v in ckpt.items():
            k = k.replace("module.", "")

            if k.startswith("backbone_encoder."):
                k = k.replace("backbone_encoder.", "")
                clean_ckpt[k] = v

            elif k.startswith("encoder."):
                k = k.replace("encoder.", "")
                clean_ckpt[k] = v

            elif k.startswith("backbone."):
                k = k.replace("backbone.", "")
                clean_ckpt[k] = v

        missing, unexpected = self.encoder.load_state_dict(
            clean_ckpt,
            strict=False
        )

        print("MOFO missing keys:", len(missing))
        print("MOFO unexpected keys:", len(unexpected))

        self.feature_dim = 512

    def forward(self, x):
        features = self.encoder(x)

        if isinstance(features, (tuple, list)):
            features = features[-1]

        if features.ndim == 4:
            features = torch.mean(features, dim=[2, 3])

        if features.ndim == 3:
            features = features[:, 0, :]

        return features


In [290]:


# ============================================================
# 7. MODEL FACTORY
# ============================================================



def build_model(model_name, num_classes):
    model_name = model_name.lower()

    if model_name == "resnet50":
        encoder = ResNet50Encoder()

    elif model_name == "efficientnetv2":
        encoder = EfficientNetV2Encoder()

    elif model_name == "dino":
        encoder = DINOEncoder()

    elif model_name == "usfmae":
        encoder = USFMAEEncoder(USF_MAE_CKPT)

    elif model_name == "ultrasam":
        encoder = UltraSAMEncoder(ULTRASAM_CKPT)

    elif model_name == "mofo":
        encoder = MOFOEncoder(MOFO_CKPT)

    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return EncoderClassifier(
        encoder=encoder,
        feature_dim=encoder.feature_dim,
        num_classes=num_classes
    )


In [291]:
# ============================================================
# 8. LINEAR PROBING AND FINE-TUNING MODES
# ============================================================

def set_training_mode(model, mode):
    if mode == "linear_probe":
        for param in model.encoder.parameters():
            param.requires_grad = False

        for param in model.classifier.parameters():
            param.requires_grad = True

    elif mode == "fine_tune":
        for param in model.parameters():
            param.requires_grad = True

    return model




def get_optimizer(model, model_name):
    params = filter(lambda p: p.requires_grad, model.parameters())

    if model_name.lower() in ["ultrasam", "dino", "usfmae"]:
        return optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

    return optim.Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)

In [292]:
# ============================================================
# 9. TRAINING ONE EPOCH
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)


In [293]:
# ============================================================
# 10. VALIDATION
# ============================================================

def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            total_loss += loss.item() * images.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    try:
        auc = roc_auc_score(
            all_labels,
            np.array(all_probs),
            multi_class="ovr",
            average="macro"
        )
    except Exception:
        auc = 0.0

    return {
        "loss": total_loss / len(loader.dataset),
        "acc": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall": recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "auc": auc
    }

In [294]:
def train_one_fold(model_name, mode, train_idx, val_idx, fold_id):
    print(f"\nModel: {model_name} | Mode: {mode} | Fold: {fold_id}")

    train_base = FolderFetalPlaneDataset(IMAGE_ROOT, transform=train_transform)
    val_base = FolderFetalPlaneDataset(IMAGE_ROOT, transform=val_transform)

    train_dataset = Subset(train_base, train_idx)
    val_dataset = Subset(val_base, val_idx)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    model = build_model(model_name, NUM_CLASSES)
    model = set_training_mode(model, mode)
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(model, model_name)

    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=7,
        gamma=0.1
    )

    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics = evaluate(model, val_loader, criterion)

        scheduler.step()

        print(
            f"Epoch [{epoch + 1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} "
            f"Train Acc: {train_acc:.4f} "
            f"Val Acc: {val_metrics['acc']:.4f} "
            f"Val F1: {val_metrics['f1']:.4f} "
            f"Val AUC: {val_metrics['auc']:.4f}"
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_weights = copy.deepcopy(model.state_dict())

    os.makedirs("checkpoints", exist_ok=True)

    save_path = os.path.join(
        "checkpoints",
        f"{model_name}_{mode}_fold{fold_id}.pth"
    )

    torch.save(best_weights, save_path)

    return best_val_acc, save_path


In [295]:
def run_cross_validation(model_name, mode):
    base_dataset = FolderFetalPlaneDataset(IMAGE_ROOT, transform=None)

    labels = [sample[1] for sample in base_dataset.samples]
    groups = [extract_patient_id(sample[0]) for sample in base_dataset.samples]

    if len(set(groups)) >= NUM_FOLDS:
        splitter = StratifiedGroupKFold(
            n_splits=NUM_FOLDS,
            shuffle=True,
            random_state=42
        )
        splits = splitter.split(base_dataset.samples, labels, groups)
    else:
        splitter = StratifiedKFold(
            n_splits=NUM_FOLDS,
            shuffle=True,
            random_state=42
        )
        splits = splitter.split(base_dataset.samples, labels)

    fold_results = []
    checkpoint_paths = []

    for fold_id, (train_idx, val_idx) in enumerate(splits, start=1):
        best_acc, ckpt_path = train_one_fold(
            model_name=model_name,
            mode=mode,
            train_idx=train_idx,
            val_idx=val_idx,
            fold_id=fold_id
        )

        fold_results.append(best_acc)
        checkpoint_paths.append(ckpt_path)

    return float(np.mean(fold_results)), checkpoint_paths


In [296]:
# ============================================================
# MAIN
# ============================================================

def main():
    models_to_test = [
        "resnet50",
        "efficientnetv2",
        "dino",
        "usfmae",
        "mofo",
        "ultrasam"
    ]

    modes = [
        "linear_probe",
        "fine_tune"
    ]

    results = []

    for model_name in models_to_test:
        for mode in modes:
            print("\n" + "=" * 80)
            print(f"Running {model_name} with {mode}")
            print("=" * 80)

            mean_cv_acc, checkpoint_paths = run_cross_validation(
                model_name=model_name,
                mode=mode
            )

            result = {
                "model": model_name,
                "mode": mode,
                "mean_cv_acc": mean_cv_acc,
                "checkpoints": checkpoint_paths
            }

            results.append(result)

            pd.DataFrame(results).to_csv("benchmark_results.csv", index=False)

    print("\nFinal Results:")
    print(pd.DataFrame(results))


if __name__ == "__main__":
    main()


Running resnet50 with linear_probe
Classes: {'Fetal abdomen': 0, 'Fetal brain': 1, 'Fetal femur': 2, 'Fetal thorax': 3, 'Maternal cervix': 4, 'Other': 5}
Total images: 12441

Model: resnet50 | Mode: linear_probe | Fold: 1
Classes: {'Fetal abdomen': 0, 'Fetal brain': 1, 'Fetal femur': 2, 'Fetal thorax': 3, 'Maternal cervix': 4, 'Other': 5}
Total images: 12441
Classes: {'Fetal abdomen': 0, 'Fetal brain': 1, 'Fetal femur': 2, 'Fetal thorax': 3, 'Maternal cervix': 4, 'Other': 5}
Total images: 12441


KeyboardInterrupt: 

In [ ]:
print("IMAGE_ROOT:", IMAGE_ROOT)
print("Exists:", os.path.exists(IMAGE_ROOT))
print("Folders:", os.listdir(IMAGE_ROOT))

IMAGE_ROOT: C:\Users\Welcome\OneDrive\Dokumen\BenchMarking_Model\Image_F
Exists: True
Folders: ['abdomen', 'Fetal brain', 'Fetal femur', 'Fetal thorax', 'Maternal cervix', 'Other']


In [ ]:
for class_name in os.listdir(IMAGE_ROOT):
    class_path = os.path.join(IMAGE_ROOT, class_name)

    if os.path.isdir(class_path):
        print("\nClass:", class_name)
        print(os.listdir(class_path)[:10])


Class: abdomen
['Fetal abdomen']

Class: Fetal brain
['Fetal brain']

Class: Fetal femur
['Fetal femur']

Class: Fetal thorax
['Fetal thorax']

Class: Maternal cervix
['Maternal cervix']

Class: Other
['Other']


In [ ]:
dataset = FolderFetalPlaneDataset(
    IMAGE_ROOT,
    transform=None
)

print(dataset.class_to_idx)
print(len(dataset))

Classes: {'abdomen': 0, 'Fetal brain': 1, 'Fetal femur': 2, 'Fetal thorax': 3, 'Maternal cervix': 4, 'Other': 5}
Total images: 0
{'abdomen': 0, 'Fetal brain': 1, 'Fetal femur': 2, 'Fetal thorax': 3, 'Maternal cervix': 4, 'Other': 5}
0
